# PTB-XL Multi-Label ECG Classification — Ablation Study

**Course:** CS-598 Deep Learning for Healthcare  
**Dataset:** PTB-XL (PhysioNet / CinC Challenge 2020, v1.0.2)  
**Model:** SparcNet (dense-block 1-D CNN designed for biosignals)

---

## Background & Motivation

PTB-XL is the largest publicly available clinical 12-lead ECG dataset, containing
21,837 recordings from 18,885 patients at 500 Hz (≈ 10 s per recording).
Each recording is annotated with one or more *SNOMED-CT* codes.

We frame ECG diagnosis as **multi-label classification**: given a signal
$X \in \mathbb{R}^{C \times T}$ ($C=12$ leads, $T$ time-steps), predict a
binary label vector $y \in \{0,1\}^K$ for $K$ diagnostic classes.

### Mathematical Framing

| Symbol | Meaning |
|--------|---------|
| $C = 12$ | ECG leads |
| $T$ | Time-steps: **1 000** at 100 Hz or **5 000** at 500 Hz |
| $K$ | Label classes: **5** (superdiagnostic) or **27** (diagnostic) |
| $f_\theta$ | SparcNet backbone |

**Forward pass:**
$$\hat{y} = \sigma\!\left(f_\theta(X)\,W^\top + b\right) \in [0,1]^K$$

**Training loss (Binary Cross-Entropy per label):**
$$\mathcal{L}_{\text{BCE}} = -\frac{1}{K}\sum_{k=1}^{K}\left[y_k\log\hat{y}_k + (1-y_k)\log(1-\hat{y}_k)\right]$$

**Evaluation — macro-averaged ROC-AUC:**
$$\overline{\text{AUC}} = \frac{1}{K}\sum_{k=1}^{K}\int_0^1 \text{TPR}_k(t)\,d\,\text{FPR}_k(t)$$

**Evaluation — macro-averaged F1 (threshold = 0.5):**
$$\overline{F_1} = \frac{1}{K}\sum_{k=1}^{K}\frac{2\,\text{TP}_k}{2\,\text{TP}_k + \text{FP}_k + \text{FN}_k}$$

---

## Ablation Design

We vary two axes simultaneously (as done in Strodthoff *et al.* 2020):

| Config | `label_type` | `sampling_rate` | $K$ | $T$ |
|--------|-------------|-----------------|-----|-----|
| **A** (baseline) | superdiagnostic | 100 Hz | 5 | 1 000 |
| **B** | superdiagnostic | 500 Hz | 5 | 5 000 |
| **C** | diagnostic | 100 Hz | 27 | 1 000 |
| **D** | diagnostic | 500 Hz | 27 | 5 000 |

Holding the model architecture and hyper-parameters **constant** across all
four configurations isolates the effect of (a) label granularity and (b)
temporal resolution on downstream performance.

**Hypothesis:**
* Finer label granularity (27 classes) is a harder task → lower absolute AUC.
* Higher temporal resolution (500 Hz) provides more information → higher AUC
  at the cost of increased model input size and training time.

## 0. Environment Setup

Install dependencies if running on a fresh Colab runtime.

In [ ]:
# Uncomment the lines below to install on Colab / a fresh environment
# !pip install pyhealth scipy wfdb --quiet
# !pip install git+https://github.com/sunlabuiuc/PyHealth.git --quiet

import sys
print(f'Python {sys.version}')

import torch
print(f'PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 1. Dataset Path

Point `PTBXL_ROOT` to the `training/ptb-xl/` sub-directory of the
PhysioNet Challenge 2020 download (v1.0.2).  
It should contain group sub-directories `g1/`, `g2/`, …, `g22/`, each
holding pairs of WFDB files (`.hea` header + `.mat` signal matrix).

```
training/ptb-xl/
  g1/
    HR00001.hea
    HR00001.mat
    ...
  g2/ ...
  ...
  g22/
```

In [ ]:
import os
from pathlib import Path

# -----------------------------------------------------------------------
# EDIT THIS to point to your local copy of the PTB-XL data
# -----------------------------------------------------------------------
PTBXL_ROOT = str(
    Path("../classification-of-12-lead-ecgs-the-physionetcomputing-in-cardiology-challenge-2020-1.0.2/training/ptb-xl")
    .resolve()
)

if not Path(PTBXL_ROOT).exists():
    raise FileNotFoundError(
        f"PTB-XL root not found: {PTBXL_ROOT}\n"
        "Please set PTBXL_ROOT to the training/ptb-xl/ directory."
    )

print(f'PTB-XL root: {PTBXL_ROOT}')
n_groups = len([d for d in Path(PTBXL_ROOT).iterdir() if d.is_dir() and d.name.startswith('g')])
print(f'Found {n_groups} group directories')

## 2. Shared Imports

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import roc_auc_score, f1_score

from pyhealth.datasets import PTBXLDataset, split_by_patient, get_dataloader
from pyhealth.tasks import PTBXLMultilabelClassification
from pyhealth.models import SparcNet
from pyhealth.trainer import Trainer
from pyhealth.metrics import multilabel_metrics_fn

## 3. Hyper-parameters

Following the grid-search described in the project paper, we fix the
best-found hyper-parameters for all four ablation runs so that the only
difference is the task configuration.

In [ ]:
# Training hyper-parameters (fixed across all ablation configs)
BATCH_SIZE   = 64    # best setting from grid search
LEARNING_RATE = 1e-3 # best setting from grid search
EPOCHS       = 5     # increase to 20–30 for full reproduction
SPLIT        = [0.7, 0.1, 0.2]  # train / val / test
MONITOR      = 'roc_auc_macro'  # PyHealth trainer monitor key

# Use dev=True to cap the dataset at ~1 000 patients for a quick smoke test.
# Set DEV_MODE=False for the full 21 837-recording experiment.
DEV_MODE = True

print(f'Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}  |  Epochs: {EPOCHS}')
print(f'Dev mode: {DEV_MODE}')

## 4. Load the PTBXLDataset (shared)

The `PTBXLDataset` parses every `.hea` header, extracts patient metadata
and SNOMED-CT codes, and writes a compact `ptbxl-pyhealth.csv` index
file on the first run.  Subsequent runs load from the parquet cache.

In [ ]:
base_dataset = PTBXLDataset(
    root=PTBXL_ROOT,
    dev=DEV_MODE,
)
base_dataset.stats()

## 5. Ablation Configurations

Define all four task variants covering the $2 \times 2$ ablation grid.

In [ ]:
ABLATION_CONFIGS = [
    {
        'name': 'A — superdiagnostic / 100 Hz (baseline)',
        'label_type': 'superdiagnostic',
        'sampling_rate': 100,
        'n_classes': 5,
        'T': 1000,
    },
    {
        'name': 'B — superdiagnostic / 500 Hz',
        'label_type': 'superdiagnostic',
        'sampling_rate': 500,
        'n_classes': 5,
        'T': 5000,
    },
    {
        'name': 'C — diagnostic (27-class) / 100 Hz',
        'label_type': 'diagnostic',
        'sampling_rate': 100,
        'n_classes': 27,
        'T': 1000,
    },
    {
        'name': 'D — diagnostic (27-class) / 500 Hz',
        'label_type': 'diagnostic',
        'sampling_rate': 500,
        'n_classes': 27,
        'T': 5000,
    },
]

print('Ablation configurations:')
for cfg in ABLATION_CONFIGS:
    print(f"  {cfg['name']}  →  K={cfg['n_classes']}, T={cfg['T']}")

## 6. Training Loop

For each configuration we:

1. **Define task** — `PTBXLMultilabelClassification(label_type, sampling_rate)`
2. **Apply task** — `base_dataset.set_task(task)` → `SampleDataset`
3. **Split** — 70 % train / 10 % val / 20 % test (by patient to avoid leakage)
4. **Instantiate SparcNet** — initialised from the `SampleDataset`
5. **Train** with `Trainer`, monitoring macro ROC-AUC on the validation set
6. **Evaluate** on the held-out test set: macro ROC-AUC + macro F1

In [ ]:
results = []

for cfg in ABLATION_CONFIGS:
    print('\n' + '='*70)
    print(f"Config: {cfg['name']}")
    print(f"  label_type={cfg['label_type']}, sampling_rate={cfg['sampling_rate']} Hz")
    print(f"  K={cfg['n_classes']} classes, T={cfg['T']} time-steps per lead")
    print('='*70)

    # ------------------------------------------------------------------
    # 6.1 Task + SampleDataset
    # ------------------------------------------------------------------
    task = PTBXLMultilabelClassification(
        label_type=cfg['label_type'],
        sampling_rate=cfg['sampling_rate'],
    )
    sample_ds = base_dataset.set_task(task)
    print(f'  Total ML samples: {len(sample_ds)}')

    sample = sample_ds[0]
    print(f'  signal shape : {tuple(sample["signal"].shape)}')
    print(f'  labels       : {sample["labels"]}')

    # ------------------------------------------------------------------
    # 6.2 Train / val / test split (by patient → no data leakage)
    # ------------------------------------------------------------------
    train_ds, val_ds, test_ds = split_by_patient(sample_ds, SPLIT)
    print(f'  Train/Val/Test samples: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}')

    train_loader = get_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = get_dataloader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = get_dataloader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

    # ------------------------------------------------------------------
    # 6.3 Model — SparcNet
    # SparcNet is a DenseNet-style 1-D CNN originally designed for EEG
    # seizure/sleep classification.  It handles variable-length 1-D signal
    # input and is well-suited for 12-lead ECG of the same length per batch.
    # ------------------------------------------------------------------
    model = SparcNet(dataset=sample_ds)

    # ------------------------------------------------------------------
    # 6.4 Train
    # ------------------------------------------------------------------
    trainer = Trainer(
        model=model,
        device=DEVICE,
        enable_logging=False,
        metrics=['roc_auc_macro', 'f1_macro'],
    )

    t0 = time.time()
    trainer.train(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        optimizer_class=torch.optim.Adam,
        optimizer_params={'lr': LEARNING_RATE},
        epochs=EPOCHS,
        monitor=MONITOR,
    )
    elapsed = time.time() - t0
    print(f'  Training time: {elapsed:.1f} s')

    # ------------------------------------------------------------------
    # 6.5 Evaluate on test set
    # ------------------------------------------------------------------
    test_metrics = trainer.evaluate(test_loader)
    roc_auc = test_metrics.get('roc_auc_macro', float('nan'))
    f1      = test_metrics.get('f1_macro',      float('nan'))

    print(f'  Test ROC-AUC (macro): {roc_auc:.4f}')
    print(f'  Test F1      (macro): {f1:.4f}')

    results.append({
        'config':        cfg['name'],
        'label_type':    cfg['label_type'],
        'sampling_rate': cfg['sampling_rate'],
        'K':             cfg['n_classes'],
        'T':             cfg['T'],
        'roc_auc_macro': roc_auc,
        'f1_macro':      f1,
        'train_time_s':  elapsed,
    })

## 7. Results Summary

In [ ]:
results_df = pd.DataFrame(results)
display_cols = ['config', 'K', 'T', 'roc_auc_macro', 'f1_macro', 'train_time_s']
print(results_df[display_cols].to_string(index=False))

## 8. Visualisation — Ablation Results

Bar charts comparing macro ROC-AUC and macro F1 across the four configs.

In [ ]:
# Short labels for the x-axis
short_labels = ['A\n(super/100Hz)', 'B\n(super/500Hz)', 'C\n(diag/100Hz)', 'D\n(diag/500Hz)']
auc_vals = results_df['roc_auc_macro'].tolist()
f1_vals  = results_df['f1_macro'].tolist()

x = np.arange(len(short_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_auc = ax.bar(x - width/2, auc_vals, width, label='ROC-AUC (macro)', color='steelblue')
bars_f1  = ax.bar(x + width/2, f1_vals,  width, label='F1 (macro)',      color='coral')

ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=11)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_ylabel('Score', fontsize=12)
ax.set_title('PTB-XL Multi-Label Ablation: SparcNet (ROC-AUC & F1 by Config)', fontsize=13)
ax.legend(fontsize=11)
ax.bar_label(bars_auc, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(bars_f1,  fmt='%.3f', padding=3, fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('ptbxl_ablation_results.png', dpi=150)
plt.show()
print('Figure saved to ptbxl_ablation_results.png')

## 9. Analysis & Findings

### Effect of Label Granularity

Comparing configs **A vs C** (both at 100 Hz): moving from the 5-class
**superdiagnostic** vocabulary to the 27-class **diagnostic** vocabulary
increases classification difficulty because:

* Rare classes have far fewer positive examples, making gradient updates noisy.
* The larger output head must learn $K = 27$ independent sigmoid thresholds.
* Macro averaging penalises poor performance on rare labels equally.

Formally, the expected macro-AUC satisfies
$$\overline{\text{AUC}}_{27} \leq \overline{\text{AUC}}_{5}$$
when the 27-class problem is strictly harder per class.

### Effect of Sampling Rate

Comparing configs **A vs B** (both superdiagnostic): at 500 Hz ($T = 5000$)
the model receives 5× more temporal resolution per lead.  This allows the
model to detect high-frequency features (notches, fragmented QRS) that are
aliased away at 100 Hz.  However:

* Input size grows by 5×, substantially increasing memory and training time.
* SparcNet's DenseNet architecture uses successive max-pooling and transition
  layers, so the *effective receptive field* scales with $T$; the model
  may not fully exploit the extra resolution within 5 epochs.

### Trade-off

Config **B** (superdiagnostic / 500 Hz) is expected to achieve the highest
absolute AUC if sufficient epochs are used, while Config **D**
(diagnostic / 500 Hz) is the most challenging in both accuracy and
compute cost.

These findings closely mirror the ablation tables in Strodthoff *et al.* (2021),
where superdiagnostic tasks consistently outperform the fine-grained ones and the
500 Hz models narrow the gap only when trained for ≥ 100 epochs.

## 10. References

1. Wagner, P. *et al.* (2020). PTB-XL, a large publicly available electrocardiography dataset.
   *Scientific Data* 7, 154. https://doi.org/10.1038/s41597-020-0495-6

2. Reyna, M.A. *et al.* (2020). Will Two Do? Varying Dimensions in Electrocardiography:
   The PhysioNet/Computing in Cardiology Challenge 2020. *CinC 2020*.

3. Strodthoff, N. *et al.* (2021). Deep Learning for ECG Analysis: Benchmarks and Insights
   from PTB-XL. *IEEE JBHI* 25, 1519–1528.

4. Jing, J. *et al.* (2023). Development of Expert-Level Classification of Seizures and
   Rhythmic and Periodic Patterns During EEG Interpretation. *Neurology* 100, e1750–e1762.

5. Zhao, M. *et al.* (2024). PyHealth: A Deep Learning Toolkit for Healthcare Predictive
   Modeling. *arXiv:2401.06284*.